# EXIST 2026 — Task 3 — **BASELINE do sistema submetido**, no split congelado

Cópia de `Thesis_video_fixed.ipynb`. Existe para uma finalidade única: produzir o
**ponto de comparação** contra `Thesis_video_improved.ipynb`.

## A única mudança de tratamento: nenhuma

O modelo, as perdas, o treino e a avaliação são **bit a bit os do sistema submetido**:
`cross_attn`, `bio_ln`, `max_length=128`, SWA a partir da época 5, fusão 2688,
15 épocas, `warmup_ratio`. Nada disso foi tocado — é exatamente o que queremos medir.

**O que mudou é o instrumento de medida, não o objeto medido:**

| # | Mudança | Por quê |
|---|---|---|
| 1 | split vem de `Dataset/val_split_42.json` em vez de `train_test_split(test_size=0.1, random_state=SEED)` | sem isto a comparação é impossível — ver abaixo |
| 2 | `SEED = 42` (era 2025) | com o split vindo do artefato, `SEED` passa a controlar **só** init + ordem dos dados; 42 pareia com `INIT_SEED=42` do notebook melhorado |
| 3 | `val_df.to_json("./Dataset/val_seed_{SEED}.json")` **removido** | escreveria um arquivo com cara de artefato de split que **não é** o congelado — armadilha de nome parecido |
| 4 | `record_fingerprint()` persiste em `fingerprints.json` e compara sozinho | o original só imprimia; comparar na scrollback não escala |
| 5 | curva por época gravada em `epoch_metrics.csv` | é o dado que você quer para a comparação |

## Por que não dava para simplesmente rodar o `Thesis_video_fixed.ipynb`

O original sorteia o próprio split: `test_size=0.1`, `random_state=SEED` →
**2 271 treino / 253 val**. O sistema melhorado usa o split congelado →
**2 019 treino / 505 val**. São sorteios independentes. Medido:

```
novo val ∩ treino ANTIGO :  454 / 505  (89.9 %)   <- CONTAMINADO
val antigo ∩ treino NOVO :  202 / 253  (79.8 %)   <- CONTAMINADO
```

**89,9 % do conjunto de validação novo foi visto em treino pelo modelo antigo.**
Comparar as métricas dos dois seria comparar um modelo avaliado em dados que nunca viu
contra outro avaliado em dados que memorizou. E os conjuntos nem têm o mesmo tamanho
(253 vs 505), então nem "aproximadamente comparável" salva.

O split é o instrumento de medida. Para atribuir a diferença às correções, ele tem de ser
**o mesmo dos dois lados**. É a única variável que precisa ser igualada — e é por isso
que ela muda aqui.

## ⚠ Limite dos fingerprints — ler antes de comparar hashes

Os fingerprints verificam **reprodutibilidade dentro de um notebook**: rode duas vezes
com a mesma seed, os hashes têm de bater. Eles **nunca** verificam equivalência entre os
dois notebooks — arquiteturas diferentes criam módulos diferentes, que consomem o RNG de
forma diferente, logo os hashes **têm** de divergir. Comparar o hash daqui com o do
notebook melhorado não ensina nada.

## ⚠ Orçamento de épocas

Este notebook roda **15 épocas + SWA a partir da 5**, como o sistema submetido. O
melhorado roda 6 (provisório) sem SWA. **Parte de qualquer delta seria orçamento de
época, não correção.** Por isso a curva por época é gravada: a comparação honesta é feita
depois, contra a época que o sistema melhorado acabar usando, dizendo de qual época veio
cada número.

---
# EXIST 2026 Task 3 — Multimodal Sexism Detection (Videos)
---

Architecture:
  - VideoMAE (video)      → 768-dim, attention-pooled over time
  - Whisper encoder feats (audio) → 1280-dim, projected to 768
  - XLM-RoBERTa-large (text)      → 1024-dim CLS
  - Biosignal features (ET + HR + viewer demographics) → 128-dim

Multi-task heads:
  - Task A → binary sexism (BCEWithLogitsLoss, soft float targets)
  - Task B → 3-way intention (NONE / JUDGEMENTAL / DIRECT, soft CE)
  - Task C → 6-way multi-label categories (BCEWithLogitsLoss, soft floats)

Key fixes vs previous version:
  1.  Fusion layer: removed bogus 3rd arg to nn.Linear
  2.  Bio MLP: bumped to 128-dim output (better balance vs text/video/audio)
  3.  Frame count: standardized on NUM_FRAMES=16 everywhere
  4.  SWA: fixed fractional-epoch comparison
  5.  load_best_model_at_end stays False (would conflict with SWA — see notes)
  6.  compute_metrics: defensive prediction unpacking
  7.  Bio: LayerNorm added after viewer-baseline subtraction
  8.  Modality dropout: 0.05 (was 0.15, way too aggressive)
  9.  Video pooling: learned attention pooling (was mean — destroys temporal order)
  10. Audio projection: MLP + LayerNorm (was bare Linear)
  11. Per-task loss logging via custom Trainer subclass
  12. Threshold tuning for Task C moved OUT of training-time eval
      (use tune_task_c_thresholds() once on val after training, then pass
       thresholds to test-set evaluation)
  13. Comments and shape annotations made consistent throughout
---

In [ ]:
# ─── Label schemas ──────────────────────────────────────────────────────────
LABELS_2_1 = ['YES', 'NO']
LABELS_2_2 = ['NONE', 'JUDGEMENTAL', 'DIRECT']
LABELS_2_3 = ['NONE', 'IDEOLOGICAL-INEQUALITY', 'MISOGYNY-NON-SEXUAL-VIOLENCE',
               'SEXUAL-VIOLENCE', 'OBJECTIFICATION', 'STEREOTYPING-DOMINANCE']

In [ ]:
import torch
import pandas as pd
import numpy as np
import re
import os
import json
import tempfile
import random
import torch.nn as nn
import torch.nn.functional as F
import emoji
from wordsegment import segment
from PIL import Image
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import f1_score
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from transformers import (
    TrainingArguments,
    Trainer,
    TrainerCallback,
    XLMRobertaTokenizer,
    XLMRobertaModel,
    VideoMAEModel,
    EarlyStoppingCallback,
    CLIPProcessor,
    CLIPVisionModel,
    set_seed,
)
from peft import LoraConfig, get_peft_model
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils
import whisper
import torchaudio

In [ ]:
# ─── Reproducibility ────────────────────────────────────────────────────────
import hashlib

# cuBLAS reads this when its handle is created (first GPU op); must be set
# before any CUDA use to take effect.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"


def set_all_seeds(seed: int):
    """Set every RNG that affects training reproducibility and force
    deterministic CUDA kernels.

    Call again right before any step whose randomness must be reproducible
    (weight init, training) — it resets the generators to the same point
    regardless of what ran before.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)

    # Without these, cuDNN picks algorithms by runtime benchmark (varies per
    # run) and CUDA uses atomic-add kernels that sum floats in arbitrary
    # order — rounding drift that compounds over epochs into different
    # final weights.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    # Require a deterministic implementation of EVERY op; raises RuntimeError
    # naming the culprit op otherwise (better than silent irreproducibility).
    torch.use_deterministic_algorithms(True)


def model_fingerprint(model):
    """SHA-256 (16 hex) over all weights — equal fingerprints mean
    bit-identical weights. Compare the printed value across runs."""
    digest = hashlib.sha256()
    for name, param in sorted(model.state_dict().items()):
        digest.update(name.encode())
        digest.update(param.detach().cpu().numpy().tobytes())
    return digest.hexdigest()[:16]


### Fingerprint persistente

O original imprimia o fingerprint (células 31 e 53) mas não o guardava, então a
verificação de reprodutibilidade dependia de comparar scrollback entre duas sessões.
Esta função grava em `RUN_DIR/fingerprints.json` e compara sozinha na segunda run.

É a **mesma implementação** que `video_core.record_fingerprint` — para que os dois
notebooks sejam verificados pelo mesmo procedimento.

In [ ]:
def record_fingerprint(run_dir, tag, model):
    '''Persiste o fingerprint e compara com o valor gravado para a mesma tag.

    tag e tipicamente "init" ou "final". Rodar o notebook duas vezes com o mesmo SEED
    tem de imprimir "identico a run anterior" nas duas tags — e isso e a prova de que o
    seeding funciona.
    '''
    path = os.path.join(run_dir, "fingerprints.json")
    current = model_fingerprint(model)

    previous = {}
    if os.path.exists(path):
        with open(path) as f:
            previous = json.load(f)

    if tag in previous:
        if previous[tag] == current:
            print(f"[fingerprint] {tag}: {current}  OK identico a run anterior")
        else:
            print(f"[fingerprint] {tag}: {current}  MISMATCH "
                  f"(run anterior: {previous[tag]}) -- a run NAO e reprodutivel")
    else:
        print(f"[fingerprint] {tag}: {current}  (primeira run; nada a comparar)")

    previous[tag] = current
    with open(path, "w") as f:
        json.dump(previous, f, indent=2)
    return current

In [ ]:
# No original: SEED = 2025, e ele controlava split + init + ordem dos dados ao mesmo
# tempo (OPT-06). Aqui o split vem do artefato congelado, então SEED controla apenas
# init dos pesos novos + ordem dos dados — o mesmo papel que INIT_SEED tem no notebook
# melhorado. 42 para parear com a run INIT_SEED=42 de lá.
SEED = 42
set_all_seeds(SEED)
os.chdir("/home/a67606/Thesis/EXIST_VIDEOS")
# Diretório desta run. Um por seed, para que rodar de novo compare fingerprints em vez
# de sobrescrever em silêncio.
RUN_DIR = f"./runs_baseline/seed_{SEED}"
os.makedirs(RUN_DIR, exist_ok=True)
print(f"SEED = {SEED} (init + ordem dos dados) | RUN_DIR = {RUN_DIR}")

In [ ]:
# ─── Global config constants ────────────────────────────────────────────────
# Use everywhere — single source of truth for dimensions and sizes.
NUM_FRAMES      = 16          # VideoMAE-base expects 16 frames
FRAME_H = FRAME_W = 224
NUM_CHANNELS    = 3

VIDEO_DIM       = 768         # VideoMAE-base hidden size
AUDIO_RAW_DIM   = 1280        # Whisper-large encoder hidden size
AUDIO_PROJ_DIM  = 768
TEXT_DIM        = 1024        # XLM-RoBERTa-large hidden size

BIO_RAW_DIM     = 16          # eye-tracking + heart-rate features per user
BIO_INPUT_DIM   = BIO_RAW_DIM + 1   # +1 for CVPC
BIO_HIDDEN_DIM  = 128         # bumped from 32 to balance vs text=1024/video=768/audio=768

VIEWER_EMB_DIM  = 32
VIEWER_PROJ_DIM = BIO_RAW_DIM  # must match bio_vec dim for subtraction

FUSION_INPUT_DIM = VIDEO_DIM + AUDIO_PROJ_DIM + TEXT_DIM + BIO_HIDDEN_DIM   # 2688
FUSION_HIDDEN_DIM = 1024

NUM_TASK_A = 1
NUM_TASK_B = len(LABELS_2_2)   # 3
NUM_TASK_C = len(LABELS_2_3)   # 6

# Viewer ID → index mapping (fixed panel of 10)
VIEWER_TO_IDX = {
    "EN1_2": 0, "EN2_2": 1, "EN3_2": 2, "EN4_2": 3, "EN5_2": 4,
    "ES1_2": 5, "ES2_2": 6, "ES3_2": 7, "ES4_2": 8, "ES5_2": 9,
}
NUM_VIEWERS = len(VIEWER_TO_IDX)

# RT imputation values per language (from Day 1 analysis)
RT_IMPUTE = {"en": 29.0, "es": 17.5}
RT_CLIP   = 120.0
THRESHOLD_GRID = np.arange(0.05, 0.65, 0.05).tolist()

In [ ]:
# ─── Frame loader ───────────────────────────────────────────────────────────
def load_frames(frames_dir: str, item_id: str) -> torch.Tensor:
    """
    Load pre-extracted video frames tensor.
    Returns [NUM_FRAMES, NUM_CHANNELS, FRAME_H, FRAME_W] = [16, 3, 224, 224] float.
    Falls back to zeros if file missing.
    """
    path = Path(frames_dir) / f"{item_id}.pt"
    try:
        frames = torch.load(path, weights_only=True)
        return frames.float()
    except Exception:
        print(f"[WARN] Frames not found for {item_id}, using zeros")
        return torch.zeros(NUM_FRAMES, NUM_CHANNELS, FRAME_H, FRAME_W,
                           dtype=torch.float32)

In [ ]:
# ─── Audio loader ───────────────────────────────────────────────────────────
def load_audio_features(audio_dir: str, item_id: str) -> torch.Tensor:
    """
    Load pre-extracted Whisper encoder features (pre-pooled, [1280] float).
    Falls back to zeros if file missing.
    """
    path = Path(audio_dir) / f"{item_id}.pt"
    try:
        features = torch.load(path, weights_only=True)
        return features.float()
    except Exception:
        print(f"[WARN] Audio features not found for {item_id}, using zeros")
        return torch.zeros(AUDIO_RAW_DIM, dtype=torch.float32)

In [ ]:
def safe_float(val, default=0.0):
    """Convert to float safely — handles None, NaN, non-numeric."""
    try:
        return float(val)
    except (TypeError, ValueError):
        return default

In [ ]:
# ─── Bio vector construction ────────────────────────────────────────────────
def build_bio_vector(
    sensorial: dict,
    lang: str,
    scaler,                   # fitted RobustScaler from training set
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Extract and preprocess biosignal features from sensorial dict.

    Returns:
        bio_vec:  [16] averaged ET+HR across 2 users (robust-scaled)
        cvpc:     [1]  cosine similarity between the 2 users' raw vectors
        rt_norm:  [1]  mean reaction time (clipped, scaled)
        fix_norm: [1]  mean fixation count (scaled)
        user_ids: [2]  viewer indices into the embedding table
    """
    users     = sensorial["users"]                    # e.g. ["ES1_2", "ES3_2"]
    et_data   = sensorial["modalities"]["ET"]["by_user"]
    hr_data   = sensorial["modalities"]["HR"]["by_user"]

    user_vecs = []
    user_ids  = []
    rt_vals   = []
    fix_vals  = []

    for user in users:
        et = et_data[user]
        hr = hr_data[user]

        # Reaction time: clip outliers + impute NaN
        rt = et.get("reaction_time", None)
        if rt is None or safe_float(rt) != safe_float(rt):  # NaN check
            rt = RT_IMPUTE[lang]
        rt = min(safe_float(rt), RT_CLIP)
        rt_vals.append(rt)

        fix = safe_float(et.get("fixations_count"), 0.0)
        fix_vals.append(fix)

        vec = np.array([
            safe_float(et.get("3d_eye_states_pupil diameter left [mm]_mean")),
            safe_float(et.get("3d_eye_states_pupil diameter left [mm]_std")),
            safe_float(et.get("3d_eye_states_pupil diameter right [mm]_mean")),
            safe_float(et.get("3d_eye_states_pupil diameter right [mm]_std")),
            safe_float(et.get("blinks_count")),
            safe_float(et.get("blinks_duration_mean_ns")),
            safe_float(et.get("fixations_count")),
            safe_float(et.get("fixations_duration_mean_ns")),
            safe_float(et.get("fixations_duration_std_ns")),
            safe_float(et.get("saccades_count")),
            safe_float(et.get("saccades_duration_mean_ns")),
            rt,
            safe_float(hr.get("garmin_hr_mean")),
            safe_float(hr.get("garmin_hr_std")),
            safe_float(hr.get("garmin_hr_max")),
            safe_float(hr.get("garmin_hr_min")),
        ], dtype=np.float32)

        user_vecs.append(vec)
        user_ids.append(VIEWER_TO_IDX[user])

    v1, v2     = user_vecs[0], user_vecs[1]
    v1_scaled  = scaler.transform(v1.reshape(1, -1)).flatten()   # [16]
    v2_scaled  = scaler.transform(v2.reshape(1, -1)).flatten()   # [16]

    # CVPC: cosine similarity on standardised vectors
    dot   = np.dot(v1_scaled, v2_scaled)
    norms = (np.linalg.norm(v1_scaled) * np.linalg.norm(v2_scaled)) + 1e-8
    cvpc  = float(dot / norms)

    # Average across the 2 standardised vectors → [16]
    bio_scaled = np.mean([v1_scaled, v2_scaled], axis=0)

    rt_mean  = float(np.mean(rt_vals))
    fix_mean = float(np.mean(fix_vals))

    return (
        torch.tensor(bio_scaled,           dtype=torch.float32),  # [16]
        torch.tensor([cvpc],               dtype=torch.float32),  # [1]
        torch.tensor([rt_mean],            dtype=torch.float32),  # [1]
        torch.tensor([fix_mean],           dtype=torch.float32),  # [1]
        torch.tensor(user_ids,             dtype=torch.long),     # [2]
    )

In [ ]:
# ─── Soft label builders ────────────────────────────────────────────────────
def build_soft_label_a(labels: list[str]) -> torch.Tensor:
    """Task A binary sexism. Proportion of YES votes; UNKNOWN excluded.
    Returns -100.0 scalar sentinel if all UNKNOWN."""
    valid = [l for l in labels if l != "UNKNOWN"]
    if len(valid) == 0:
        return torch.tensor(-100.0, dtype=torch.float32)
    soft = sum(1 for l in valid if l == "YES") / len(valid)
    return torch.tensor(soft, dtype=torch.float32)

In [ ]:
def build_soft_label_b(labels: list[str]) -> torch.Tensor:
    """Task B (NONE/JUDGEMENTAL/DIRECT). '-' → NONE. UNKNOWN excluded.
    Returns [-1.0, -1.0, -1.0] sentinel if all UNKNOWN."""
    remapped = []
    for l in labels:
        if l == "UNKNOWN":
            continue
        remapped.append("NONE" if l == "-" else l)

    if len(remapped) == 0:
        return torch.full((len(LABELS_2_2),), -1.0, dtype=torch.float32)

    counts = {lbl: 0 for lbl in LABELS_2_2}
    for l in remapped:
        if l in counts:
            counts[l] += 1

    soft = np.array([counts[lbl] / len(remapped) for lbl in LABELS_2_2], dtype=np.float32)
    return torch.tensor(soft, dtype=torch.float32)

In [ ]:
def build_soft_label_c(labels: list[list[str]]) -> torch.Tensor:
    """Task C multi-label, 6 categories. '-' → NONE. UNKNOWN excluded.
    Returns [-1.0, ...] sentinel if all UNKNOWN."""
    remapped = []
    for ann_labels in labels:
        cleaned = []
        for lbl in ann_labels:
            if lbl == "UNKNOWN":
                continue
            cleaned.append("NONE" if lbl == "-" else lbl)
        if cleaned:
            remapped.append(cleaned)

    if len(remapped) == 0:
        return torch.full((len(LABELS_2_3),), -1.0, dtype=torch.float32)

    counts = {lbl: 0 for lbl in LABELS_2_3}
    for ann_labels in remapped:
        for lbl in ann_labels:
            if lbl in counts:
                counts[lbl] += 1

    soft = np.array([counts[lbl] / len(remapped) for lbl in LABELS_2_3], dtype=np.float32)
    return torch.tensor(soft, dtype=torch.float32)

In [ ]:
def tokenize_text(text: str, tokenizer, max_length: int) -> dict:
    """Tokenize transcript text with padding and truncation."""
    return tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )

In [ ]:
def collect_bio_vectors(dataframe: pd.DataFrame) -> np.ndarray:
    """Flatten per-user bio vectors across the whole DF, for fitting the scaler."""
    vectors = []
    for _, row in dataframe.iterrows():
        sensorial = row["sensorial"]
        lang      = row["lang"]
        et_data   = sensorial["modalities"]["ET"]["by_user"]
        hr_data   = sensorial["modalities"]["HR"]["by_user"]
        for user in sensorial["users"]:
            et = et_data[user]
            hr = hr_data[user]
            rt = safe_float(et.get("reaction_time"), default=float("nan"))
            if rt != rt:  # NaN check
                rt = RT_IMPUTE[lang]
            rt = min(rt, RT_CLIP)
            vec = np.array([
                safe_float(et.get("3d_eye_states_pupil diameter left [mm]_mean")),
                safe_float(et.get("3d_eye_states_pupil diameter left [mm]_std")),
                safe_float(et.get("3d_eye_states_pupil diameter right [mm]_mean")),
                safe_float(et.get("3d_eye_states_pupil diameter right [mm]_std")),
                safe_float(et.get("blinks_count")),
                safe_float(et.get("blinks_duration_mean_ns")),
                safe_float(et.get("fixations_count")),
                safe_float(et.get("fixations_duration_mean_ns")),
                safe_float(et.get("fixations_duration_std_ns")),
                safe_float(et.get("saccades_count")),
                safe_float(et.get("saccades_duration_mean_ns")),
                rt,
                safe_float(hr.get("garmin_hr_mean")),
                safe_float(hr.get("garmin_hr_std")),
                safe_float(hr.get("garmin_hr_max")),
                safe_float(hr.get("garmin_hr_min")),
            ], dtype=np.float32)
            vectors.append(vec)
    return np.stack(vectors)

In [ ]:
# ─── Dataset ────────────────────────────────────────────────────────────────
class EXISTVideoDataset(Dataset):
    def __init__(
        self,
        dataframe,
        tokenizer,
        scaler,
        frames_dir: str,
        audio_dir: str,
        max_length: int = 128,
    ):
        self.tokenizer  = tokenizer
        self.scaler     = scaler
        self.frames_dir = frames_dir
        self.audio_dir  = audio_dir
        self.max_length = max_length

        self.item_ids   = dataframe["id_EXIST"].values
        self.langs      = dataframe["lang"].values
        self.texts      = dataframe["text"].values
        self.sensorials = dataframe["sensorial"].values
        self.labels_a   = dataframe["labels_task3_1"].values
        self.labels_b   = dataframe["labels_task3_2"].values
        self.labels_c   = dataframe["labels_task3_3"].values

    def __len__(self) -> int:
        return len(self.item_ids)

    def __getitem__(self, idx: int) -> dict:
        item_id   = self.item_ids[idx]
        lang      = self.langs[idx]
        text      = self.texts[idx]
        sensorial = self.sensorials[idx]

        frames         = load_frames(self.frames_dir, item_id)
        audio_features = load_audio_features(self.audio_dir, item_id)
        tokenized      = tokenize_text(text, self.tokenizer, self.max_length)

        bio_vec, cvpc, rt_norm, fix_norm, user_ids = build_bio_vector(
            sensorial, lang, self.scaler
        )

        label_a = build_soft_label_a(self.labels_a[idx])
        label_b = build_soft_label_b(self.labels_b[idx])
        label_c = build_soft_label_c(self.labels_c[idx])

        return {
            "frames":         frames,                          # [16, 3, 224, 224]
            "audio":          audio_features,                  # [1280]
            "input_ids":      tokenized["input_ids"].squeeze(0),
            "attention_mask": tokenized["attention_mask"].squeeze(0),
            "bio_vec":        bio_vec,                         # [16]
            "cvpc":           cvpc,                            # [1]
            "rt_norm":        rt_norm,                         # [1]
            "fix_norm":       fix_norm,                        # [1]
            "user_ids":       user_ids,                        # [2]
            "labels_a":       label_a,                         # scalar
            "labels_b":       label_b,                         # [3]
            "labels_c":       label_c,                         # [6]
        }

In [ ]:
# ─── Data loading and split ─────────────────────────────────────────────────
# MUDANCA 1 (a unica de instrumento): o split vem do artefato CONGELADO, o mesmo que
# Thesis_video_improved.ipynb carrega. Sem isto os dois sistemas seriam avaliados em
# conjuntos diferentes (253 vs 505 amostras, 89,9% de contaminacao) e a comparacao nao
# significaria nada.
#
# NAO reintroduzir train_test_split aqui. O split e o instrumento de medida; se ele
# muda entre os dois lados, o delta deixa de ser atribuivel as correcoes.
SPLIT_ARTIFACT = "./Dataset/val_split_42.json"

assert os.path.exists(SPLIT_ARTIFACT), (
    f"{SPLIT_ARTIFACT} nao encontrado. O artefato do split e criado localmente e "
    "Dataset/ NAO e sincronizado com a VM -- copiar antes de rodar. "
    "Sem ele nao ha comparacao possivel com o sistema melhorado.")

full_df = pd.read_json('./Dataset/training/EXIST2026_training.json').transpose()

with open(SPLIT_ARTIFACT) as f:
    frozen = json.load(f)

ids = full_df["id_EXIST"].astype(str).values
train_ids = set(frozen["train_ids"])
val_ids   = set(frozen["val_ids"])

missing = (train_ids | val_ids) - set(ids)
assert not missing, (
    f"O artefato refere {len(missing)} id_EXIST ausentes do dataframe. "
    "Artefato e dados divergiram -- investigar, NAO recalcular o split por cima.")

mask_train = np.array([i in train_ids for i in ids])
mask_val   = np.array([i in val_ids   for i in ids])

train_df = full_df[mask_train].reset_index(drop=True)
val_df   = full_df[mask_val].reset_index(drop=True)

print(f"[split] carregado de {SPLIT_ARTIFACT} "
      f"(split_seed={frozen['split_seed']}, estratificacao={frozen['stratify']})")
print(f"[split] {len(train_df)} treino / {len(val_df)} val")
assert len(train_df) == frozen["n_train"] and len(val_df) == frozen["n_val"], \
    "contagens divergem do artefato"

# MUDANCA 3: val_df.to_json("./Dataset/val_seed_{SEED}.json") foi REMOVIDO de proposito.
# Gravaria um arquivo com nome quase igual ao do artefato congelado, sem ser ele --
# exatamente o tipo de near-miss que alguem carrega por engano seis meses depois.

In [ ]:
train_bio_matrix = collect_bio_vectors(train_df)
scaler = RobustScaler().fit(train_bio_matrix)
import pickle, os
os.makedirs(f"./best_model_seed_{SEED}", exist_ok=True)
with open(f"./best_model_seed_{SEED}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [ ]:
tokenizer = XLMRobertaTokenizer.from_pretrained("xlm-roberta-large")

train_dataset = EXISTVideoDataset(train_df, tokenizer, scaler,
                                  "frames/train/", "audio_features/train/")
val_dataset   = EXISTVideoDataset(val_df,   tokenizer, scaler,
                                  "frames/train/", "audio_features/train/")

In [ ]:
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

In [ ]:
sample = train_dataset[0]
for k, v in sample.items():
    print(f"{k:20s}: {v.shape if hasattr(v, 'shape') else v}")

# VideoMAE with LoRA config

In [ ]:
# Re-seed right before weight init: LoRA A matrices are randomly initialized,
# so init must not depend on how much RNG earlier cells consumed.
set_all_seeds(SEED)

videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")

lora_cfg = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value"],
)
videomae = get_peft_model(videomae, lora_cfg)
videomae.print_trainable_parameters()


# XLM-RoBERTa LoRA config

In [ ]:
set_all_seeds(SEED)  # re-seed before LoRA init (same reason as VideoMAE cell)

xlmr = XLMRobertaModel.from_pretrained("xlm-roberta-large")

lora_cfg_text = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["query", "value"],
)
xlmr = get_peft_model(xlmr, lora_cfg_text)
xlmr.print_trainable_parameters()


# SWA Callback

In [ ]:
class SWACallback(TrainerCallback):
    """Stochastic Weight Averaging callback for HuggingFace Trainer.

    Maintains a running average of model weights starting from `swa_start_epoch`.
    At the end of training, swaps the trainer's model with the averaged one
    and updates BatchNorm statistics if any BN layers exist.

    NOTE: This conflicts with `load_best_model_at_end=True`. HF Trainer calls
    `on_train_end` BEFORE loading the best checkpoint, so the best checkpoint
    would overwrite SWA weights. Keep `load_best_model_at_end=False` when SWA
    is enabled.
    """
    def __init__(self, swa_start_epoch: int = 8, swa_lr: float = None):
        self.swa_start_epoch = swa_start_epoch
        self.swa_lr = swa_lr
        self.swa_model = None
        self.swa_n = 0

    def on_train_begin(self, args, state, control, model=None, **kwargs):
        self.swa_model = AveragedModel(model)
        device = next(model.parameters()).device
        self.swa_model = self.swa_model.to(device)
        print(f"[SWA] Initialised. Averaging starts at epoch {self.swa_start_epoch}.")

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if state.epoch is None:
            return
        if state.epoch + 1e-6 >= self.swa_start_epoch:
            self.swa_model.update_parameters(model)
            self.swa_n += 1
            print(f"[SWA] Epoch {state.epoch:.2f}: averaged model updated "
                  f"(n={self.swa_n})")

    def on_train_end(self, args, state, control, model=None,
                     train_dataloader=None, **kwargs):
        if self.swa_n == 0:
            print("[SWA] No epochs averaged — SWA disabled in final model.")
            return

        print(f"[SWA] Training complete. Averaged over {self.swa_n} epochs.")
        # update_bn is a no-op if no BN layers exist. Uncomment if you add BN:
        # update_bn(train_dataloader, self.swa_model,
        #           device=next(model.parameters()).device)
        model.load_state_dict(self.swa_model.module.state_dict())
        print("[SWA] Averaged weights loaded into model. Final eval uses SWA.")

# Model

In [ ]:
class TemporalAttentionPool(nn.Module):
    """Lightweight learned attention pooling over the temporal dim.

    Replaces `mean(dim=1)` which discards temporal ordering. Each frame token
    gets a scalar attention weight; the output is a weighted sum across tokens.
    """
    def __init__(self, hidden_dim: int, attn_dim: int = 128):
        super().__init__()
        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, attn_dim),
            nn.Tanh(),
            nn.Linear(attn_dim, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, T, D]
        logits  = self.attn(x)                          # [B, T, 1]
        weights = torch.softmax(logits, dim=1)          # [B, T, 1]
        pooled  = (weights * x).sum(dim=1)              # [B, D]
        return pooled


class EXISTVideoModel(nn.Module):
    def __init__(self, videomae, xlmr):
        super().__init__()

        self.videomae = videomae
        self.xlmr     = xlmr


        self.video_pool = TemporalAttentionPool(VIDEO_DIM)


        self.audio_proj = nn.Sequential(
            nn.Linear(AUDIO_RAW_DIM, AUDIO_PROJ_DIM),
            nn.GELU(),
            nn.LayerNorm(AUDIO_PROJ_DIM),
        )

        self.viewer_emb  = nn.Embedding(NUM_VIEWERS, VIEWER_EMB_DIM)

        self.viewer_proj = nn.Sequential(
            nn.Linear(VIEWER_EMB_DIM, VIEWER_PROJ_DIM),
            nn.Tanh(),
        )
        self.bio_ln  = nn.LayerNorm(BIO_RAW_DIM)
        self.bio_mlp = nn.Sequential(
            nn.Linear(BIO_INPUT_DIM, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, BIO_HIDDEN_DIM),
            nn.GELU(),
        )


        # FUSION_INPUT_DIM = 768 + 768 + 1024 + 128 = 2688
        self.fusion = nn.Sequential(
            nn.Linear(FUSION_INPUT_DIM, FUSION_HIDDEN_DIM),
            nn.GELU(),
            nn.Dropout(0.1),
        )

        self.head_a = nn.Linear(FUSION_HIDDEN_DIM, NUM_TASK_A)
        self.head_b = nn.Linear(FUSION_HIDDEN_DIM, NUM_TASK_B)
        self.head_c = nn.Linear(FUSION_HIDDEN_DIM, NUM_TASK_C)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=TEXT_DIM,
            num_heads=8,
            batch_first=True,
            dropout=0.1,
        )
        self.cross_proj = nn.Linear(VIDEO_DIM, TEXT_DIM)
        self.cross_norm = nn.LayerNorm(TEXT_DIM)

    def gradient_checkpointing_enable(self, **kwargs):
        self.videomae.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )
        self.xlmr.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": False}
        )

    def _zeros(self, B, D, device, dtype):
        """Helper for dropped-modality fallback tensors."""
        return torch.zeros(B, D, device=device, dtype=dtype)

    def forward(
            self,
            frames,
            audio,
            input_ids,
            attention_mask,
            bio_vec,
            cvpc,
            rt_norm,
            fix_norm,
            user_ids,
            labels_a=None,
            labels_b=None,
            labels_c=None,

            use_bio=True,
            use_viewer_emb=True,
            modality_dropout_prob=0.05,
            **kwargs,
        ):
            B      = frames.shape[0]
            device = frames.device
            ref_dtype = self.fusion[0].weight.dtype


            drop_video = drop_audio = drop_text = drop_bio = False
            if self.training and modality_dropout_prob > 0:
                r = torch.rand(1).item()
                p = modality_dropout_prob
                if   r < p:                   drop_video = True
                elif r < 2 * p:               drop_audio = True
                elif r < 3 * p:               drop_text  = True
                elif r < 4 * p and use_bio:   drop_bio   = True

            # ── Video ─────────────────────────────────────────────────────
            if drop_video:
                vid_feat = self._zeros(B, VIDEO_DIM, device, ref_dtype)
            else:
                vid_out  = self.videomae(pixel_values=frames).last_hidden_state  # [B, T, 768]
                vid_feat = self.video_pool(vid_out)                              # [B, 768]

            # ── Audio ─────────────────────────────────────────────────────
            if drop_audio:
                aud_feat = self._zeros(B, AUDIO_PROJ_DIM, device, ref_dtype)
            else:
                aud_feat = self.audio_proj(audio)                                # [B, 768]

            # ── Text ──────────────────────────────────────────────────────
            if drop_text:
                txt_feat = self._zeros(B, TEXT_DIM, device, ref_dtype)
            else:
                txt_feat = self.xlmr(
                    input_ids=input_ids,
                    attention_mask=attention_mask
                ).last_hidden_state[:, 0, :]                                     # [B, 1024]

            # ── Bio ───────────────────────────────────────────────────────
            if use_bio and not drop_bio:
                if use_viewer_emb:
                    # Per-viewer physiological baseline (Tanh-bounded to ±1).
                    base1 = self.viewer_proj(self.viewer_emb(user_ids[:, 0]))    # [B, 16]
                    base2 = self.viewer_proj(self.viewer_emb(user_ids[:, 1]))    # [B, 16]
                    bio_centered = bio_vec - (base1 + base2) / 2                 # [B, 16]
                else:
                    bio_centered = bio_vec                                       # [B, 16]

                bio_norm  = self.bio_ln(bio_centered)                            # [B, 16]
                bio_input = torch.cat([bio_norm, cvpc], dim=-1)                  # [B, 17]
                bio_feat  = self.bio_mlp(bio_input)                              # [B, BIO_HIDDEN_DIM]
            else:
                # Bio ablation OR modality dropout fired on bio.
                bio_feat = self._zeros(B, BIO_HIDDEN_DIM, device, ref_dtype)

            # ── Fusion ────────────────────────────────────────────────────
            vid_proj = self.cross_proj(vid_feat).unsqueeze(1)
            txt_q    = txt_feat.unsqueeze(1)
            cross_out, _ = self.cross_attn(query=txt_q, key=vid_proj, value=vid_proj)
            txt_feat = self.cross_norm(txt_feat + cross_out.squeeze(1))
            fused = self.fusion(torch.cat([vid_feat, aud_feat, txt_feat, bio_feat], dim=-1))  # [B, 2688]

            # ── Task logits ───────────────────────────────────────────────
            logits_a = self.head_a(fused).squeeze(-1)   # [B]
            logits_b = self.head_b(fused)               # [B, 3]
            logits_c = self.head_c(fused)               # [B, 6]

            loss = None
            loss_components = None
            if labels_a is not None:
                loss, loss_components = compute_loss(
                    logits_a, logits_b, logits_c,
                    labels_a, labels_b, labels_c,
                    return_components=True,
                )
                if loss.dim() != 0:
                    loss = loss.mean()

            if loss is not None:
                # Stash per-task losses for the custom Trainer to log.
                self._last_loss_components = loss_components
                return (loss, logits_a, logits_b, logits_c)
            else:
                return (logits_a, logits_b, logits_c)

# Loss function

In [ ]:
def loss_task_a(logits_a, labels_a):
    """BCE with logits, soft float targets. Sentinel -100.0 masks empties."""
    mask = (labels_a != -100.0).float()
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits_a.device)
    loss = F.binary_cross_entropy_with_logits(
        logits_a, labels_a.clamp(0, 1), reduction="none"
    )
    return (loss * mask).sum() / mask.sum()


def loss_task_b(logits_b, labels_b):
    """Soft cross-entropy (CrossEntropyLoss doesn't support soft targets).
    Sentinel -1.0 in first column marks empty rows."""
    mask = (labels_b[:, 0] != -1.0).float()
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits_b.device)
    log_pb = F.log_softmax(logits_b, dim=-1)
    loss   = -(labels_b.clamp(0, 1) * log_pb).sum(dim=-1)
    return (loss * mask).sum() / mask.sum()


def loss_task_c(logits_c, labels_c):
    """BCE per-class (categories don't sum to 1). Sentinel -1.0 masks empties."""
    mask = (labels_c[:, 0] != -1.0).float()
    if mask.sum() == 0:
        return torch.tensor(0.0, device=logits_c.device)
    loss = F.binary_cross_entropy_with_logits(
        logits_c, labels_c.clamp(0, 1), reduction="none"
    ).mean(dim=-1)
    return (loss * mask).sum() / mask.sum()


def compute_loss(logits_a, logits_b, logits_c,
                 labels_a, labels_b, labels_c,
                 return_components: bool = False):
    la = loss_task_a(logits_a, labels_a)
    lb = loss_task_b(logits_b, labels_b)
    lc = loss_task_c(logits_c, labels_c)
    total = la + 0.7 * lb + 0.7 * lc
    if return_components:
        return total, {"loss_a": la.detach(), "loss_b": lb.detach(), "loss_c": lc.detach()}
    return total

In [ ]:
set_all_seeds(SEED)  # re-seed so EXISTVideoModel's new layers init identically every run

model = EXISTVideoModel(videomae, xlmr)
print("Initial weights fingerprint:", model_fingerprint(model))
record_fingerprint(RUN_DIR, "init", model)

In [ ]:
def get_optimizer(model):
    lora_params = [p for n, p in model.named_parameters()
                   if "lora" in n and p.requires_grad]
    viewer_params = [p for n, p in model.named_parameters()
                     if "viewer_emb" in n and p.requires_grad]
    cross_params = [p for n, p in model.named_parameters()
                    if any(x in n for x in ["cross_attn", "cross_proj", "cross_norm"])
                    and p.requires_grad]
    cross_ids = {id(p) for p in cross_params}
    base_params = [p for n, p in model.named_parameters()
                   if "lora" not in n
                   and "viewer_emb" not in n
                   and not any(x in n for x in ["cross_attn", "cross_proj", "cross_norm"])
                   and p.requires_grad]

    optimizer = torch.optim.AdamW([
        {"params": lora_params,   "lr": 1e-4,  "weight_decay": 0.01},
        {"params": viewer_params, "lr": 1e-3,  "weight_decay": 0.0},
        {"params": base_params,   "lr": 1e-3,  "weight_decay": 0.05},
        {"params": cross_params,  "lr": 1e-4,  "weight_decay": 0.01},  # same as LoRA
    ])
    return optimizer

In [ ]:
training_args = TrainingArguments(
    output_dir=f"./checkpoints_seed_{SEED}",
    num_train_epochs=15,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,

    # memory
    bf16=True,
    fp16=False,
    gradient_checkpointing=True,
    dataloader_num_workers=2,
    load_best_model_at_end=False,


    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    warmup_ratio=0.1,
    lr_scheduler_type="cosine",

    remove_unused_columns=False,

    report_to="none",
    logging_steps=10,
    seed=SEED,
    data_seed=SEED,
)

# Task C threshold search (post-training only)

In [ ]:
def _search_thresholds_task_c(probs_c, targets_c, ids_c, temp_dir):
    """Greedy coordinate search over per-class thresholds for Task C hard ICMNorm.

    Optimizes one class at a time, holding the others fixed.
    12 candidates × 6 classes = 72 PyEvALL calls.

    Returns: (best_thresholds: list[float], final_scores: dict)

    IMPORTANT: Only run this ONCE on validation data after training is complete.
    Running every epoch during training causes validation overfitting (the
    thresholds end up tuned to noise in the dev set).
    """
    best_thresholds = [0.5] * len(LABELS_2_3)

    for class_idx in range(len(LABELS_2_3)):
        best_t   = best_thresholds[class_idx]
        best_icm = -np.inf

        for t in THRESHOLD_GRID:
            candidate = best_thresholds.copy()
            candidate[class_idx] = float(t)

            scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                  temp_dir=temp_dir, thresholds_c=candidate)
            icm = scores.get('ICMNorm', -np.inf)
            if icm > best_icm:
                best_icm = icm
                best_t   = float(t)

        best_thresholds[class_idx] = best_t

    final_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                temp_dir=temp_dir, thresholds_c=best_thresholds)
    return best_thresholds, final_scores


def tune_task_c_thresholds(trainer, eval_dataset, save_path: str = None):
    print("[Threshold tuning] Generating predictions on validation set...")
    pred_out = trainer.predict(eval_dataset)
    preds    = pred_out.predictions
    labels   = pred_out.label_ids

    # Defensive unpacking — see _unpack_predictions for full handling.
    logits_a, logits_b, logits_c = _unpack_predictions(preds)
    labels_a, labels_b, labels_c = labels

    logits_c = np.asarray(logits_c)
    labels_c = np.asarray(labels_c)

    mask_c = labels_c[:, 0] != -1.0
    if not mask_c.any():
        print("[Threshold tuning] No valid Task C samples — skipping.")
        return None

    probs_c   = 1 / (1 + np.exp(-logits_c[mask_c]))
    targets_c = labels_c[mask_c]
    ids_c     = [str(i) for i in range(len(probs_c))]

    with tempfile.TemporaryDirectory() as temp_dir:
        best_thresholds, final_scores = _search_thresholds_task_c(
            probs_c, targets_c, ids_c, temp_dir
        )

    result = {
        "thresholds": best_thresholds,
        "scores":     final_scores,
        "label_order": LABELS_2_3,
    }

    print("[Threshold tuning] Done. Best per-class thresholds:")
    for lbl, t in zip(LABELS_2_3, best_thresholds):
        print(f"  {lbl:30s}: {t:.2f}")
    print(f"[Threshold tuning] Final Task C hard ICMNorm: "
          f"{final_scores.get('ICMNorm', float('nan')):.4f}")

    if save_path:
        with open(save_path, "w") as f:
            json.dump(result, f, indent=2)
        print(f"[Threshold tuning] Saved to {save_path}")

    return result

# PyEvALL helpers

In [ ]:
def _format_for_pyevall(probs, sample_ids, task, evaluation, thresholds_c=None):
    """Convert probability arrays to PyEvALL JSON format.

    thresholds_c: optional list of 6 floats for Task C hard mode.
                  If None, defaults to 0.5 for all classes.
    """
    out = []
    for i, sid in enumerate(sample_ids):
        entry = {"test_case": "EXIST2025", "id": str(sid)}

        if task == '2_1':
            p_no  = float(probs[i][0])
            p_yes = float(probs[i][1])
            if evaluation == 'hard':
                entry["value"] = "YES" if p_yes >= p_no else "NO"
            else:
                entry["value"] = {"YES": p_yes, "NO": p_no}

        elif task == '2_2':
            label_map = {"NONE": "NO", "JUDGEMENTAL": "JUDGEMENTAL", "DIRECT": "DIRECT"}
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_2)}
            if evaluation == 'hard':
                entry["value"] = max(relevant, key=relevant.get)
            else:
                entry["value"] = relevant

        elif task == '2_3':
            label_map = {"NONE": "NO"}
            label_map.update({lbl: lbl for lbl in LABELS_2_3[1:]})
            relevant  = {label_map[lbl]: float(probs[i][idx])
                         for idx, lbl in enumerate(LABELS_2_3)}

            if evaluation == 'hard':
                threshs = thresholds_c if thresholds_c is not None else [0.5] * len(LABELS_2_3)
                thresh_map = {label_map[lbl]: threshs[idx]
                              for idx, lbl in enumerate(LABELS_2_3)}
                hard_labels = [lbl for lbl, p in relevant.items() if p >= thresh_map[lbl]]
                if not hard_labels:
                    hard_labels = [max(relevant, key=relevant.get)]
                entry["value"] = hard_labels
            else:
                entry["value"] = relevant

        out.append(entry)
    return out

In [ ]:
TASK_2_2_HIERARCHY = {
    "YES": ["DIRECT", "JUDGEMENTAL"],
    "NO":  []
}
TASK_2_3_HIERARCHY = {
    "YES": [
        "IDEOLOGICAL-INEQUALITY",
        "STEREOTYPING-DOMINANCE",
        "OBJECTIFICATION",
        "SEXUAL-VIOLENCE",
        "MISOGYNY-NON-SEXUAL-VIOLENCE",
    ],
    "NO": [],
}

In [ ]:
def _extract_metrics_from_report(report, metric_list):
    """Pull metric scores out of a PyEvALL report object."""
    results = {}
    if not hasattr(report, 'report') or 'metrics' not in report.report:
        return results

    metrics_dict = report.report['metrics']
    for metric in metric_list:
        if metric not in metrics_dict:
            continue
        if metrics_dict[metric].get('status') != 'OK':
            continue
        try:
            results[metric] = float(
                metrics_dict[metric]['results']['average_per_test_case']
            )
        except (KeyError, TypeError, ValueError):
            pass
    return results

In [ ]:
def _run_pyevall(probs, gold_dist, sample_ids, task, evaluation,
                 temp_dir='./pyevall_temp_train', thresholds_c=None):
    os.makedirs(temp_dir, exist_ok=True)

    preds = _format_for_pyevall(probs, sample_ids, task, evaluation,
                                 thresholds_c=thresholds_c)
    gold  = _format_for_pyevall(gold_dist, sample_ids, task, evaluation)
    pred_path = f'{temp_dir}/pred_{task}_{evaluation}.json'
    gold_path = f'{temp_dir}/gold_{task}_{evaluation}.json'
    with open(pred_path, 'w') as f: json.dump(preds, f)
    with open(gold_path, 'w') as f: json.dump(gold, f)

    if evaluation == 'hard':
        metric_list = ['ICM', 'ICMNorm', 'FMeasure']
    else:
        if task == '2_3':
            metric_list = ['ICMSoft', 'ICMSoftNorm']
        else:
            metric_list = ['ICMSoft', 'ICMSoftNorm', 'CrossEntropy']

    params = {}
    if task == '2_2':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_2_HIERARCHY
    elif task == '2_3':
        params[PyEvALLUtils.PARAM_HIERARCHY] = TASK_2_3_HIERARCHY

    try:
        evaluator = PyEvALLEvaluation()
        report    = evaluator.evaluate(pred_path, gold_path, metric_list, **params)
        return _extract_metrics_from_report(report, metric_list)
    except Exception as e:
        print(f"  ! PyEvALL failed for task {task} ({evaluation}): {e}")
        return {}

In [ ]:
def _unpack_predictions(preds):
    if isinstance(preds, tuple) and len(preds) == 3:
        return preds
    if isinstance(preds, list) and len(preds) == 3:
        return tuple(preds)
    if isinstance(preds, dict):
        try:
            return preds["logits_a"], preds["logits_b"], preds["logits_c"]
        except KeyError as e:
            raise RuntimeError(
                f"Prediction dict missing expected key {e}. Keys: {list(preds.keys())}"
            )
    if isinstance(preds, np.ndarray):
        raise RuntimeError(
            "Predictions came back as a single ndarray — expected 3-tuple "
            "(logits_a, logits_b, logits_c). Check the model forward return."
        )
    raise TypeError(f"Unexpected prediction type: {type(preds)}")


def compute_metrics(eval_pred, tune_thresholds_c: bool = False):
    logits_a, logits_b, logits_c = _unpack_predictions(eval_pred.predictions)
    labels_a, labels_b, labels_c = eval_pred.label_ids

    logits_a = np.asarray(logits_a)
    logits_b = np.asarray(logits_b)
    logits_c = np.asarray(logits_c)
    labels_a = np.asarray(labels_a)
    labels_b = np.asarray(labels_b)
    labels_c = np.asarray(labels_c)

    metrics = {}

    with tempfile.TemporaryDirectory() as temp_dir:

        # === Task A ===
        if labels_a.ndim == 2:
            labels_a = labels_a[:, 1]

        mask_a = labels_a != -100.0
        if mask_a.any():
            prob_yes  = 1 / (1 + np.exp(-logits_a[mask_a].reshape(-1)))
            probs_a   = np.stack([1 - prob_yes, prob_yes], axis=-1)
            targets_a = np.stack([1 - labels_a[mask_a], labels_a[mask_a]], axis=-1)
            ids_a     = [str(i) for i in range(len(probs_a))]

            for evaluation in ('hard', 'soft'):
                scores = _run_pyevall(probs_a, targets_a, ids_a, '2_1', evaluation,
                                      temp_dir=temp_dir)
                for k, v in scores.items():
                    metrics[f'task_a_{k.lower()}_{evaluation}'] = v

        # === Task B ===
        mask_b = labels_b[:, 0] != -1.0
        if mask_b.any():
            x         = logits_b[mask_b]
            x         = x - x.max(axis=-1, keepdims=True)
            probs_b   = np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)
            targets_b = labels_b[mask_b]
            ids_b     = [str(i) for i in range(len(probs_b))]

            for evaluation in ('hard', 'soft'):
                scores = _run_pyevall(probs_b, targets_b, ids_b, '2_2', evaluation,
                                      temp_dir=temp_dir)
                for k, v in scores.items():
                    metrics[f'task_b_{k.lower()}_{evaluation}'] = v

        # === Task C ===
        mask_c = labels_c[:, 0] != -1.0
        if mask_c.any():
            probs_c   = 1 / (1 + np.exp(-logits_c[mask_c]))
            targets_c = labels_c[mask_c]
            ids_c     = [str(i) for i in range(len(probs_c))]

            soft_scores = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'soft',
                                       temp_dir=temp_dir)
            for k, v in soft_scores.items():
                metrics[f'task_c_{k.lower()}_soft'] = v

            hard_default = _run_pyevall(probs_c, targets_c, ids_c, '2_3', 'hard',
                                        temp_dir=temp_dir)
            for k, v in hard_default.items():
                metrics[f'task_c_{k.lower()}_hard'] = v


            if tune_thresholds_c:
                best_thresholds, tuned = _search_thresholds_task_c(
                    probs_c, targets_c, ids_c, temp_dir
                )
                for k, v in tuned.items():
                    metrics[f'task_c_{k.lower()}_hard_tuned'] = v
                for j, label in enumerate(LABELS_2_3):
                    metrics[f'task_c_thresh_{label.lower().replace("-", "_")}'] = \
                        round(best_thresholds[j], 2)

    return metrics

# Sanity checks

In [ ]:
cvpc_values = [train_dataset[i]["cvpc"].item() for i in range(len(train_dataset))]
print(f"CVPC mean = {np.mean(cvpc_values):.4f}")
print(f"CVPC std  = {np.std(cvpc_values):.4f}")
print(f"CVPC range = [{min(cvpc_values):.4f}, {max(cvpc_values):.4f}]")

In [ ]:
sample_frames = train_dataset[0]["frames"]
assert sample_frames.shape == (NUM_FRAMES, NUM_CHANNELS, FRAME_H, FRAME_W), \
    f"Expected ({NUM_FRAMES}, {NUM_CHANNELS}, {FRAME_H}, {FRAME_W}), got {sample_frames.shape}"
print(f"✓ Frame shape matches: {sample_frames.shape}")

In [ ]:
def count_params(model: nn.Module) -> dict:
    total      = sum(p.numel() for p in model.parameters())
    trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return {"total": total, "trainable": trainable,
            "trainable_pct": 100 * trainable / max(total, 1)}


def forward_sanity_check(model, dataset, device="cuda"):
    """Run a single forward+backward pass and print all intermediate shapes."""
    model.eval()
    sample = dataset[0]
    batch  = {k: (v.unsqueeze(0).to(device) if torch.is_tensor(v) else v)
              for k, v in sample.items()}
    with torch.no_grad():
        out = model(**batch)
    print("─── Forward sanity check ───")
    if len(out) == 4:
        loss, la, lb, lc = out
        print(f"  loss      : {loss.item():.4f}")
    else:
        la, lb, lc = out
    print(f"  logits_a  : {tuple(la.shape)}")
    print(f"  logits_b  : {tuple(lb.shape)}")
    print(f"  logits_c  : {tuple(lc.shape)}")
    print(f"  params    : {count_params(model)}")

In [ ]:
print([n for n, _ in model.named_parameters() if "viewer" in n.lower()])
print([n for n, _ in model.named_parameters() if "video_pool" in n.lower()])

# Custom Trainer with per-task loss logging

In [ ]:
class MultiTaskTrainer(Trainer):
    """Logs per-task loss components alongside the total loss.

    This is purely observational — it does not change optimisation. Use the
    logged loss_a / loss_b / loss_c values to decide whether the fixed
    1.0/0.7/0.7 weighting is balanced or needs uncertainty weighting.
    """
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._loss_a_accum = 0.0
        self._loss_b_accum = 0.0
        self._loss_c_accum = 0.0
        self._loss_n       = 0

    def compute_loss(self, model, inputs, return_outputs=False,
                     num_items_in_batch=None):
        outputs = model(**inputs)
        loss    = outputs[0]

        # Pull stashed per-task losses from the underlying nn.Module.
        inner = model.module if hasattr(model, "module") else model
        comps = getattr(inner, "_last_loss_components", None)
        if comps is not None:
            self._loss_a_accum += float(comps["loss_a"])
            self._loss_b_accum += float(comps["loss_b"])
            self._loss_c_accum += float(comps["loss_c"])
            self._loss_n       += 1

        return (loss, outputs) if return_outputs else loss

    def log(self, logs, *args, **kwargs):
        if self._loss_n > 0 and "loss" in logs:
            logs["loss_a"] = self._loss_a_accum / self._loss_n
            logs["loss_b"] = self._loss_b_accum / self._loss_n
            logs["loss_c"] = self._loss_c_accum / self._loss_n
            self._loss_a_accum = 0.0
            self._loss_b_accum = 0.0
            self._loss_c_accum = 0.0
            self._loss_n       = 0
        super().log(logs, *args, **kwargs)

# Train

In [ ]:
optimizer    = get_optimizer(model)
swa_callback = SWACallback(swa_start_epoch=5)

trainer = MultiTaskTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
    callbacks=[swa_callback],
)

trainer.label_names = ["labels_a", "labels_b", "labels_c"]

In [ ]:
print("Parameter count:", count_params(model))

In [ ]:
train_output = trainer.train()
print(train_output)

## Curva por época — o dado da comparação

Grava `epoch_metrics.csv` no `RUN_DIR`. É este arquivo que vai ser confrontado com o do
sistema corrigido.

Ao comparar, dizer **de qual época veio cada número**. Este notebook roda 15 épocas com
SWA a partir da 5; o corrigido roda 6 sem SWA. Comparar "final contra final" misturaria
o efeito das correções com o efeito do orçamento de épocas.

Referência do finding ARCH-06, medida no sistema submetido: as 15 épocas + SWA produziam
um modelo final **pior** que a época 4/6 em todas as seis métricas (Task A ICMNorm hard
0,6199 → 0,5647). Se isso se repetir aqui, o "melhor baseline" não é a última linha.

In [ ]:
evals = [h for h in trainer.state.log_history if "eval_loss" in h]
curva = pd.DataFrame(evals)
curva.to_csv(os.path.join(RUN_DIR, "epoch_metrics.csv"), index=False)

cols = [c for c in curva.columns
        if c in ("epoch", "eval_loss") or c.startswith("eval_task")]
print(curva[cols].to_string(index=False))

best = min(evals, key=lambda h: h["eval_loss"])
print(f"\nargmin(eval_loss) = epoca {best['epoch']:.0f} "
      f"(eval_loss = {best['eval_loss']:.4f})")
print(f"\nGravado: {os.path.join(RUN_DIR, 'epoch_metrics.csv')}")

with open(os.path.join(RUN_DIR, "train_output.json"), "w") as f:
    json.dump({k: (float(v) if isinstance(v, (int, float)) else str(v))
               for k, v in train_output.metrics.items()}, f, indent=2)

In [ ]:
import safetensors.torch, os, pickle
from accelerate.utils import extract_model_from_parallel

os.makedirs(f"./best_model_seed_{SEED}_cross_att", exist_ok=True)

actual_model = extract_model_from_parallel(trainer.model)
state_dict = {k: v.cpu().clone() for k, v in actual_model.state_dict().items()}
safetensors.torch.save_file(state_dict, f"./best_model_seed_{SEED}_cross_att/model.safetensors")
tokenizer.save_pretrained(f"./best_model_seed_{SEED}_cross_att")
with open(f"./best_model_seed_{SEED}_cross_att/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print(f"Saved to ./best_model_seed_{SEED}_cross_att")
print("Final weights fingerprint:", model_fingerprint(actual_model))
record_fingerprint(RUN_DIR, "final", actual_model)

In [ ]:
# MUDANCA 6 (instrumento, nao tratamento): apontar a validacao para o diretorio
# que a celula 58 realmente escreve.
CKPT_SAVED = f"./best_model_seed_{SEED}_cross_att/model.safetensors"

# ── Checkpoint validation ─────────────────────────────────────────────────
print("Validating saved checkpoint...")

videomae_check = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
videomae_check = get_peft_model(videomae_check, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["query", "value"]
))
xlmr_check = XLMRobertaModel.from_pretrained("xlm-roberta-large")
xlmr_check = get_peft_model(xlmr_check, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["query", "value"]
))
model_check = EXISTVideoModel(videomae_check, xlmr_check)

state_dict_check = safetensors.torch.load_file(
    CKPT_SAVED  # <- MUDANCA 6: era ./best_model_seed_{SEED}/, que a gravacao
               # nunca escreve. A celula 58 grava em ./best_model_seed_{SEED}_cross_att/.
               # Por isso esta validacao NUNCA rodou no sistema submetido: crashava com
               # RuntimeError de chaves em falta (SIMP-02 / BUG-A). Agora le o que a
               # gravacao escreveu., device="cpu"
)
model_check.load_state_dict(state_dict_check)

# Compare a few key weights between saved and trained model
mismatches = 0
for key in ["fusion.0.weight", "head_a.weight", "head_b.weight", "head_c.weight"]:
    orig = actual_model.state_dict()[key].cpu()
    loaded = model_check.state_dict()[key].cpu()
    if not torch.allclose(orig, loaded):
        print(f"  MISMATCH: {key}")
        mismatches += 1
    else:
        print(f"  OK: {key}")

# Also verify it's not just random init (weights should differ from a fresh model)
fresh_videomae = VideoMAEModel.from_pretrained("MCG-NJU/videomae-base")
fresh_videomae = get_peft_model(fresh_videomae, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["query", "value"]
))
fresh_xlmr = XLMRobertaModel.from_pretrained("xlm-roberta-large")
fresh_xlmr = get_peft_model(fresh_xlmr, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["query", "value"]
))
fresh_model = EXISTVideoModel(fresh_videomae, fresh_xlmr)

trained_differs_from_fresh = not torch.allclose(
    model_check.state_dict()["head_a.weight"].cpu(),
    fresh_model.state_dict()["head_a.weight"].cpu()
)

if mismatches == 0 and trained_differs_from_fresh:
    print("\n✓ Checkpoint is valid — weights match trained model and differ from random init")
elif mismatches > 0:
    print(f"\n✗ {mismatches} weight mismatches — something went wrong with saving")
else:
    print("\n✗ Loaded weights are identical to random init — model was not saved correctly")

del model_check, fresh_model, state_dict_check

In [ ]:
final_metrics = trainer.evaluate()
print("[SWA] Final SWA model metrics:")
for k, v in sorted(final_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

# MUDANCA 7 (instrumento): persistir. No original estes numeros so existiam na
# scrollback -- e sao exatamente o baseline que queremos confrontar depois.
with open(os.path.join(RUN_DIR, "final_metrics.json"), "w") as f:
    json.dump({k: (float(v) if isinstance(v, (int, float)) else str(v))
               for k, v in final_metrics.items()}, f, indent=2)
print(f"\ngravado: {os.path.join(RUN_DIR, 'final_metrics.json')}")
print(f"gravado: {os.path.join(RUN_DIR, 'epoch_metrics.csv')}  (curva por epoca)")
print(f"gravado: {os.path.join(RUN_DIR, 'fingerprints.json')}  (init + final)")

# Post-training: Task C threshold tuning (run ONCE on val)

In [ ]:
# This is the ONLY place threshold search runs. The output thresholds are
# then frozen and applied at test time via _format_for_pyevall(thresholds_c=...).
threshold_result = tune_task_c_thresholds(
    trainer,
    val_dataset,
    # MUDANCA 8 (instrumento): gravar dentro do RUN_DIR, nao na raiz.
    # ./task_c_thresholds_seed_42.json JA EXISTE e e o artefato do sistema
    # submetido -- gravar por cima destruiria o registo sem aviso.
    save_path=os.path.join(RUN_DIR, "task_c_thresholds.json"),
)

# Ablation harness

In [ ]:
# ############################################################################
# CELULA DESATIVADA DE PROPOSITO -- NAO DESCOMENTAR SEM LER
# ############################################################################
#
# Dois motivos, ambos verificados:
#
# 1. DESTRUTIVA. `df.to_csv(f"ablation_seed_{SEED}.csv")` com SEED=42 sobrescreve
#    ./ablation_seed_42.csv, e `save_for_paper(df)` sobrescreve ablation_results.tex.
#    Esses arquivos sao o REGISTO do sistema submetido ao EXIST 2026 e nao podem ser
#    regenerados -- as predicoes oficiais vieram deles.
#
# 2. MEDE A COISA ERRADA. `run_all_ablations` avalia o MESMO modelo treinado com
#    bio+viewer, apenas zerando entradas no forward (ARCH-20). Nao sao ablacoes:
#    sao sondas de zeragem de input. E `ABLATION_CONDITIONS` contem um braco
#    fantasma -- B2_bio_viewer_no_tau e FULL_bio_viewer_tau tem dicts de flags
#    IDENTICOS (nao existe parametro use_tau; era engolido por **kwargs), e as
#    linhas do CSV saem bit a bit iguais.
#
# A tabela de ablacao defensavel sao 9 runs independentes (3 condicoes x 3 seeds),
# com as flags no CONSTRUTOR -- e vive no notebook melhorado, nao aqui.
# Este notebook tem uma finalidade unica: produzir o baseline no split congelado.
#
# import importlib
# import ablation_harness
# importlib.reload(ablation_harness)
# from ablation_harness import run_all_ablations, print_ablation_table, save_for_paper
#
# model   = trainer.model
# results = run_all_ablations(model, val_dataset, compute_metrics, device="cuda")
# df      = print_ablation_table(results)
# df.to_csv(f"ablation_seed_{SEED}.csv", index=False)
# save_for_paper(df)

print("celula de ablacao desativada -- ver comentario acima")